# 03 – Calculating peptide features

**Input:** `results/clean_peptides_clustered.csv` (from notebook 02)  
**Output:** `results/peptide_features.csv` – one row per cleaned peptide with a few sequence-based features. The CD-HIT columns from notebook 02 are carried along; notebooks 04 and 05 use only the non-redundant representatives (`is_representative`).

## Features and where their definitions come from
All features are computed from the amino-acid sequence only, with published definitions implemented in **Biopython** (Cock et al. 2009):

| Feature | Definition | Source |
|---|---|---|
| `length` | number of residues | already in the data |
| `net_charge` | charge of the peptide at **pH 7.0**, from the pK values of the charged side chains (K, R, H, D, E, C, Y) and the two chain ends: Biopython `ProteinAnalysis.charge_at_pH(7.0)` | pK values of Bjellqvist et al. (1993, 1994), as documented in Biopython's `IsoelectricPoint` module |
| `charge_per_residue` | `net_charge / length` – a *normalised* net charge, so short and long peptides can be compared | idea of a length-normalised net charge: Uversky et al. (2000) |
| `gravy` | **mean hydropathy** = average Kyte-Doolittle value of the residues (GRAVY, "grand average of hydropathy"). Positive = hydrophobic, negative = hydrophilic. Biopython `ProteinAnalysis.gravy()` | Kyte & Doolittle (1982) |
| `pct_A ... pct_Y` | percentage of each of the 20 amino acids (relative frequency of residues) | Vacic et al. (2007) use the same relative frequencies |

pH 7.0 is our choice of a neutral pH. Because the charge is calculated from the sequence alone, chemical modifications of the peptide ends (for example C-terminal amidation, which DBAASP records separately and which can strongly change activity; Pirtskhalava et al. 2016) are **not** taken into account.

In [1]:
import pandas as pd
from Bio.SeqUtils.ProtParam import ProteinAnalysis     # Biopython (Cock et al. 2009)
from Bio.SeqUtils.ProtParamData import kd              # Kyte-Doolittle hydropathy values (Kyte & Doolittle 1982)

IN = "../results/clean_peptides_clustered.csv"
OUT = "../results/peptide_features.csv"
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
PH = 7.0

clean = pd.read_csv(IN)
print(clean.shape)
clean.head(3)

(18275, 26)


,ID,name,sequence,length,length_class,origin,synthesis_type,has_D_aa,has_target_info,target_gram_pos,...,target_insect,target_nematode,target_mollicute,target_protista,target_archaea,target_mammalian_cell,cluster_id,is_representative,cluster_size,cluster_n_origins
0,11,"Cathelicidin-1, CATH-1, Fowlicidin-1",RVKRVWPLVIRTVIAGYNLYRAIKKK,26,Medium (25–50 aa),Natural,Ribosomal,False,True,1,...,0,0,0,0,0,1,C1915,True,21,2
1,14,Brevinin-2 related peptide B2RP,GIWDTIKSMGKVFAGKILQNL,21,Short (10–24 aa),Natural,Ribosomal,False,True,1,...,0,0,0,0,0,1,C2923,True,7,2
2,15,Histatin 5,DSHAKRHHGYKRKFHEKHHSHRGY,24,Short (10–24 aa),Natural,Ribosomal,False,True,1,...,0,0,0,0,0,1,C1195,False,17,2


## Feature functions
Thin wrappers around Biopython, tested on toy sequences before running on the real data.

In [2]:
def net_charge(seq, ph=PH):
    """Charge at the given pH (Bjellqvist pK values, via Biopython)."""
    return ProteinAnalysis(seq).charge_at_pH(ph)

def gravy(seq):
    """Mean Kyte-Doolittle hydropathy (via Biopython)."""
    return ProteinAnalysis(seq).gravy()

def composition(seq):
    """Percentage of each amino acid, as a dict like {'pct_A': 10.0, ...}"""
    return {f"pct_{aa}": 100 * seq.count(aa) / len(seq) for aa in AMINO_ACIDS}

# --- quick tests on toy sequences ---
assert net_charge("KKKKK") > 4.5              # five basic residues -> about +5
assert net_charge("DDDDD") < -4.5             # five acidic residues -> about -5
assert abs(net_charge("GGGG")) < 1            # no charged side chains -> only the two chain ends, close to 0
# GRAVY must equal the plain average of the Kyte-Doolittle values
for s in ["AAAAA", "KRKRK", "GLFDKW"]:
    assert abs(gravy(s) - sum(kd[a] for a in s) / len(s)) < 1e-9
assert gravy("AAAAA") == 1.8 and abs(gravy("KKKKK") + 3.9) < 1e-9
assert composition("AAGG")["pct_A"] == 50 and composition("AAGG")["pct_K"] == 0
assert abs(sum(composition("ACDEFGHIKL").values()) - 100) < 1e-9
print("toy tests passed")

toy tests passed


## Compute features

In [3]:
feat = clean.copy()
feat["net_charge"] = feat["sequence"].apply(net_charge)
feat["charge_per_residue"] = feat["net_charge"] / feat["length"]
feat["gravy"] = feat["sequence"].apply(gravy)

comp = pd.DataFrame(feat["sequence"].apply(composition).tolist())
feat = pd.concat([feat, comp], axis=1)
feat.shape

(18275, 49)

## Sanity checks

In [4]:
assert len(feat) == len(clean) and feat["ID"].is_unique
assert feat[["net_charge", "charge_per_residue", "gravy"]].notna().all().all()
assert feat["gravy"].between(-4.5, 4.5).all()          # Kyte-Doolittle values range from -4.5 to 4.5
assert (feat[[f"pct_{aa}" for aa in AMINO_ACIDS]].sum(axis=1) - 100).abs().max() < 1e-6
assert (feat["net_charge"].abs() <= feat["length"] + 2).all()   # at most one charge per residue plus the two chain ends
print("all checks passed")

all checks passed


## First look: Natural vs Synthetic
Just medians (no plots or tests yet; that is notebook 04), to see that the numbers make sense: antimicrobial peptides are expected to be positively charged (Pirtskhalava et al. 2016 also report more Lys/Arg in synthetic and more Lys in ribosomal peptides than in an average protein).

In [5]:
cols = ["length", "net_charge", "charge_per_residue", "gravy"]
nr = feat[feat["is_representative"]]
print(f"{len(feat)} peptides in total, {len(nr)} non-redundant representatives (medians below use the representatives only)")
nr.groupby("origin")[cols].median().round(2)

18275 peptides in total, 9951 non-redundant representatives (medians below use the representatives only)


,length,net_charge,charge_per_residue,gravy
origin,,,,
Natural,25.0,2.68,0.09,0.17
Synthetic,14.0,3.76,0.27,-0.35


## Save
(all peptides are saved; notebooks 04 and 05 keep only `is_representative`)

In [6]:
feat.to_csv(OUT, index=False)
print("saved", feat.shape, "->", OUT)

saved (18275, 49) -> ../results/peptide_features.csv


## Notes for the methods section
* Net charge and hydropathy are sequence-based and ignore terminal modifications, side-chain modifications and D-amino-acid stereochemistry (D-amino acids were converted to upper case in notebook 01).
* Net charge depends on the chosen pH (7.0 here) and on the pK table used by Biopython.
* Mean hydropathy (GRAVY) depends on the Kyte-Doolittle scale; other scales would give different values.

## References
* Cock PJA, et al. (2009) Biopython: freely available Python tools for computational molecular biology and bioinformatics. *Bioinformatics* 25(11):1422-1423. https://doi.org/10.1093/bioinformatics/btp163 (PMID 19304878)
* Kyte J, Doolittle RF (1982) A simple method for displaying the hydropathic character of a protein. *J Mol Biol* 157(1):105-132. https://doi.org/10.1016/0022-2836(82)90515-0 (PMID 7108955)
* Bjellqvist B, et al. (1993) The focusing positions of polypeptides in immobilized pH gradients can be predicted from their amino acid sequences. *Electrophoresis* 14(10):1023-1031. https://doi.org/10.1002/elps.11501401163 (PMID 8125050)
* Bjellqvist B, et al. (1994) Reference points for comparisons of two-dimensional maps of proteins from different human cell types defined in a pH scale where isoelectric points correlate with polypeptide compositions. *Electrophoresis* 15(3-4):529-539. https://doi.org/10.1002/elps.1150150171 (PMID 8055880)
* Uversky VN, Gillespie JR, Fink AL (2000) Why are "natively unfolded" proteins unstructured under physiologic conditions? *Proteins* 41(3):415-427. https://doi.org/10.1002/1097-0134(20001115)41:3<415::aid-prot130>3.0.co;2-7 (PMID 11025552). Used only for the idea of analysing a *normalised* net charge together with mean hydrophobicity; the exact formula here (net charge divided by length) is ours.
* Vacic V, Uversky VN, Dunker AK, Lonardi S (2007) Composition Profiler: a tool for discovery and visualization of amino acid composition differences. *BMC Bioinformatics* 8:211. https://doi.org/10.1186/1471-2105-8-211 (PMID 17578581)
* Pirtskhalava M, et al. (2016) DBAASP v.2. *Nucleic Acids Res* 44(D1):D1104-D1112. https://doi.org/10.1093/nar/gkv1174 (PMID 26578581)